# BIS walkthrough

The Bank for International Settlements is the central banks' bank, and it publishes
the cross-country series they compile for one another: policy rates, total credit to
the non-financial sector, debt service ratios, residential and commercial property
prices, effective exchange rates, and the banking, debt-securities and derivatives
statistics. No key, no registration, no quota to nurse.

The surprising part is that a BIS series has no id. It has *coordinates*. A series key
is the dataflow's dimension values joined by dots in a fixed order — `M.US` is monthly,
United States — and nothing in the tool signatures tells you what that order is or
which codes are legal. Only the dataflow's data structure does. So here discovery is
not a nicety ahead of the fetch; it is the only way to build a request at all.

The MCP server must be running (`python -m mcp_server.server`); the cells below reach
it over SSE at `MCP_URL`.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from matplotlib import pyplot
from lib import config
from utils import (
    list_mcp_tools,
    show_tool_schema,
    show_dataflows,
    dsd_id_for_flow,
    show_datastructure,
    find_codes,
    get_series,
    decode_series,
    show_series,
    plot_bis_series,
)

pyplot.style.use(config.glyfish_style)

## 1. Discovery

Three tools, and they line up one-to-one with the three questions BIS makes you answer
in order: which dataset (`bis_dataflows`), how a key in that dataset is shaped
(`bis_datastructure`), and only then the numbers (`bis_series_data`).

In [ ]:
await list_mcp_tools("bis_")

schema = await show_tool_schema("bis_series_data")

The schema is the interesting part precisely because it constrains almost nothing:
`flow` and `key` are bare strings with no enum, and `key` even defaults to `all`.
That is not an oversight — the legal values differ per dataflow, so no fixed signature
could carry them. The schema is telling a caller to go read the structure.
(`start_period` / `end_period` are SDMX period strings — `2020`, `2020-01`, `2020-Q1` —
matching the flow's own frequency.)

## 2. Finding a series

So start at the catalog rather than with a key from somewhere. Each dataflow entry also
names the data structure that defines its keys.

In [ ]:
flows = await show_dataflows()

print()
dsd_id = dsd_id_for_flow(flows, "WS_CBPOL")

Reading the `structure` urn matters because the DSD id is not derivable from the flow
id. It is *usually* the flow id with `WS_` swapped for `BIS_` (WS_TC → BIS_TOTAL_CREDIT),
but WS_CBTA is served by `CBTA`, WS_SPP by `BIS_SELECTED_PP`, and WS_CPP and WS_DPP
share a single structure, `BIS_PROP_PRICES`. Structures are objects in their own right,
not a naming convention on flows.

The structure answers both questions a key raises: the *order* of the dimensions, and
the codelist that says what each position accepts. By default the codes themselves are
left out and merely counted — `CL_BIS_UNIT` alone holds over a thousand, and a flow can
reference several such lists — so the cheap call is the one to make while shopping, and
the full one only for the flow you settled on.

In [ ]:
brief = await show_datastructure(dsd_id)
print("\nCL_FREQ, default:      ", brief["codelists"]["CL_FREQ"])

dsd = await show_datastructure(dsd_id, include_codes=True)
print("\nCL_FREQ, include_codes:", dsd["codelists"]["CL_FREQ"])

Both calls report `code_count: 8` for `CL_FREQ`; only the second fills in `codes`. The
count is what makes the default usable — you can size a codelist, and see which one
decodes which dimension, without pulling any of it.

Two dimensions here, in this order: `FREQ`, then `REF_AREA`. A key is therefore
`<frequency>.<area>`, and the two codelists below are where each half must come from.
A position may be left empty to wildcard it (`M.` — monthly, every area), take
alternatives with `+` (`M.US+GB`), and `all` stands in for the entire flow.

In [ ]:
find_codes(dsd, "FREQ")

print()
find_codes(dsd, "REF_AREA", "United")

key = "M.US"   # monthly, United States

In [ ]:
# The codelist is BIS-wide, not per flow: 239 reference areas exist across the whole
# service, but a flow publishes far fewer. A wildcarded key is the cheapest way to see
# which ones this dataflow actually carries.
published = await get_series("WS_CBPOL", "M.", start_period="2024-01")

print(len(published["series"]), "areas publish a monthly policy rate:")
print(", ".join(sorted(s["dimensions"]["REF_AREA"] for s in published["series"])))

## 3. Fetch and plot

`M.US` is the US policy rate, monthly, end of period — one of the few series where
seven decades stay legible on a single axis: 22% at the Volcker peak in December 1980,
0.125% from the end of 2008 through 2015 and again through the pandemic, and back near
3.6% today.

Values arrive as codes, not names: a series carries `REF_AREA: US` and
`UNIT_MEASURE: 368`. `decode_series` re-reads the structure with `include_codes=True`
and maps them back, which is what lets the plot label its own axis "Per cent per year"
rather than "368". The key that comes back is also longer than the one sent — BIS
repeats the attribute columns on every CSV row, so `UNIT_MEASURE` rides along in it.

In [ ]:
data = await decode_series("WS_CBPOL", key, dsd_id)

policy_rate = show_series(data)[0]

In [ ]:
plot_bis_series(policy_rate)